# The second case: marketing pushes back

**Week 1, Thursday. The afternoon's second case, 45 minutes, in pairs.** Four moves and eight lettered
choices, then the reply spoken aloud to a partner playing marketing. The brief is
`exercises/unguided/C2_W01_D04_pushback_STUDENT.md`.

> **The client asks.** "Your segment split is slicing the data until it says what you want. Exposed
> Retail-Plus members spent Rs 4,850 in August. That is far above the Rs 3,200 our unexposed customers
> averaged. The sale works." The marketing lead, Kalpa Retail

Replace each `__TODOn__` with the line you choose from the comment above it. Run All stops at the
first placeholder until you do.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

EXPOSURE = kit.load_csv("C2_W01_D04_exposure_STUDENT.csv")
CAMPAIGN = kit.load_csv("C2_W01_D04_campaigns_STUDENT.csv")[0]
PAIR = ["Retail-Plus", "Retail-Core"]


def mean(values):
    return sum(values) / len(values)


def avg(segment, flag):
    return mean([int(r["august_revenue"]) for r in EXPOSURE if r["segment"] == segment and r["exposed"] == flag])


def count(flag, segment=None):
    return sum(1 for r in EXPOSURE if r["exposed"] == flag and (segment is None or r["segment"] == segment))


print(CAMPAIGN["name"], CAMPAIGN["starts"], "to", CAMPAIGN["ends"], "at", CAMPAIGN["discount_pct"], "percent off,",
      "targeted at", CAMPAIGN["target_segment"])

## Move 1. Reproduce marketing's two numbers

A pushback is answered from the same table, so first make both numbers appear.

In [ ]:
# TODO 1. Which line reproduces marketing's Rs 4,850?
#   a) avg("Retail-Plus", "yes")
#   b) avg("Retail-Plus", "no")
#   c) avg("Retail-Core", "yes")
#   d) mean([int(r["august_revenue"]) for r in EXPOSURE])
marketing_yes = __TODO1__
# TODO 2. Which line reproduces marketing's Rs 3,200?
#   a) avg("Retail-Core", "no")
#   b) mean([int(r["august_revenue"]) for r in EXPOSURE if r["exposed"] == "no"])
#   c) avg("Retail-Plus", "no")
#   d) mean([int(r["august_revenue"]) for r in EXPOSURE if r["exposed"] == "no" and r["segment"] != "Business"])
marketing_no = __TODO2__
kit.columns(["exposed Retail-Plus", "all unexposed"], [("August revenue per customer", [marketing_yes, round(marketing_no)])],
            fmt=kit.rupees, width=560, title="Marketing's comparison, reproduced")
kit.check("marketing's Rs 4,850 is reproduced", marketing_yes == 4850, kit.rupees(marketing_yes))
kit.check("marketing's Rs 3,200 is reproduced", round(marketing_no) == 3200, kit.rupees(marketing_no))

## Move 2. Name the mismatch

Look at who sits in each of marketing's two groups before judging the gap between them.

In [ ]:
mix_yes = {s: count("yes", s) / count("yes") for s in PAIR}
mix_no = {s: count("no", s) / count("no") for s in PAIR}
kit.table(["Group", "Customers", "Retail-Plus", "Retail-Core"],
          [("exposed", count("yes"), count("yes", "Retail-Plus"), count("yes", "Retail-Core")),
           ("not exposed", count("no"), count("no", "Retail-Plus"), count("no", "Retail-Core"))],
          caption="Who is in each group")
kit.columns(PAIR, [("share of exposed", [round(100 * mix_yes[s]) for s in PAIR]),
                   ("share of not exposed", [round(100 * mix_no[s]) for s in PAIR])],
            fmt=lambda v: f"{v:.0f}%", width=560, title="The mix of the two groups")
# TODO 3. What makes marketing's comparison unfair?
#   a) "The exposure table is too small to say anything about August at all."
#   b) "August is a monsoon month, so no August comparison can ever be trusted by anyone, whatever the table says."
#   c) "It sets one segment's exposed members against all unexposed customers, mostly Retail-Core."
#   d) "Marketing rounded Rs 4,850 up from a smaller number."
mismatch = __TODO3__
print(mismatch)
kit.check("the exposed group leans more to Retail-Plus", mix_yes["Retail-Plus"] > mix_no["Retail-Plus"])
kit.check("most unexposed customers are Retail-Core", mix_no["Retail-Core"] > 0.5, f"{mix_no['Retail-Core']:.0%}")

## Move 3. Rebuild it like for like

Two fair versions: inside each segment, and the whole exposed group put into the unexposed group's mix.

In [ ]:
# TODO 4. What is the like-for-like comparison for Retail-Plus?
#   a) blend_no
#   b) avg("Retail-Core", "no")
#   c) avg("Retail-Plus", "yes")
#   d) avg("Retail-Plus", "no")
plus_like = avg("Retail-Plus", "yes") / __TODO4__ - 1
core_like = avg("Retail-Core", "yes") / avg("Retail-Core", "no") - 1
# TODO 5. Which weights put the exposed group into the unexposed group's mix?
#   a) mix_yes[s]
#   b) 0.5
#   c) mix_no[s]
#   d) count("yes", s) / len(EXPOSURE) * 2
reweighted_yes = sum(__TODO5__ * avg(s, "yes") for s in PAIR)
blend_no = marketing_no
fair_lift = reweighted_yes / blend_no - 1
kit.columns(PAIR, [("not exposed", [avg(s, "no") for s in PAIR]), ("exposed", [avg(s, "yes") for s in PAIR])],
            fmt=kit.rupees, width=560, title="Like for like: each segment, exposed against not exposed")
kit.bridge(("unexposed, as mixed", round(blend_no)), [("the discount, same mix", round(reweighted_yes - blend_no))],
           end_label="exposed, same mix", lo=2500, title="Same mix, only the discount differs (axis from Rs 2,500)")
print(f"Retail-Plus {plus_like:+.1%}, Retail-Core {core_like:+.1%}, same-mix blend {fair_lift:+.1%}")
# TODO 6. Which caveat does the exposure table itself force?
#   a) "The table covers only August, so it says nothing about how customers will behave in the Diwali weeks."
#   b) "Retail-Core should never have been in the campaign comparison at all."
#   c) "The campaign ran for fifteen days, which is too short to measure anything."
#   d) "Every customer in a group carries the same August figure, so the spread is unseen."
table_caveat = __TODO6__
kit.check("inside Retail-Plus the exposed spent 3 percent less", round(plus_like, 3) == -0.03, f"{plus_like:+.1%}")
kit.check("in the same mix the discount is a 3 percent fall", round(fair_lift, 3) == -0.03, f"{fair_lift:+.1%}")
kit.check("every exposed Retail-Plus customer carries one figure",
          len({r["august_revenue"] for r in EXPOSURE if r["segment"] == "Retail-Plus" and r["exposed"] == "yes"}) == 1)

## Move 4. What the discount cost, and the reply

Monday's rule: a discount of d needs volume to rise by 1 / (1 minus d), less one, just to stand still.

In [ ]:
off = int(CAMPAIGN["discount_pct"]) / 100
# TODO 7. What volume lift does 15 percent off need just to stand still?
#   a) 0.15
#   b) 1 / (1 - 0.15) - 1
#   c) 1 - 0.15
#   d) 0.15 / (1 + 0.15) + 0.15 / 2
needed = __TODO7__
# TODO 8. Which reply holds the line in the meeting?
#   a) "Inside each segment exposed customers spent 3 percent less; the blend rose because more Retail-Plus got it. Let us hold back a slice at Diwali."
#   b) "You are wrong about the sale."
#   c) "Fine, repeat it for Diwali, since the blended number is positive, marketing owns the campaign budget anyway, and the split is only one way of reading it."
#   d) "We cannot say anything."
reply = __TODO8__
print(f"{CAMPAIGN['discount_pct']} percent off needs {needed:.1%} more volume to stand still")
print(reply)
kit.flow(["reproduce\nRs 4,850 and Rs 3,200", "mismatch\nwho is in each group", "like for like\n3 percent less",
          "reply\nhold back a slice"], lit=3, title="The pair's four moves")
kit.check("15 percent off needs 17.6 percent more volume", round(needed, 3) == 0.176, f"{needed:.3f}")
kit.check("the reply offers the Diwali test", "hold back" in reply)

In [ ]:
kit.check_summary()
print("Post the eight letters in order, then say the reply aloud to your partner playing marketing.")